# JALAAKAR — validation notebook

**Owner:** Dev B · run this before every merge to `main`, and again before the Sat 20:00 freeze.

The hard checks live in `tools/validate.py` so they can also run headless in one command:

```bash
python tools/validate.py          # exit 0 = safe to freeze
```

This notebook runs the same checks and then gives you the interactive views —
the ones where you have to *look* at the curve and decide whether it is
hydrologically believable.

In [ ]:
import sys, os
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

# to inspect the synthetic scratch DB instead of the real one:
# os.environ['JALAAKAR_DB'] = 'data/test_jalaakar.db'

import pandas as pd, matplotlib.pyplot as plt
from ingest.db import connect, read, summary, config

cfg = config()
pd.set_option('display.width', 140)
with connect(readonly=True) as con:
    display(summary(con))

## 1. The full check suite

In [ ]:
!python tools/validate.py

## 2. The monsoon must be visible

If the spike is not clearly June–September, the dates or units are wrong.
Catch it now, not on Monday.

In [ ]:
with connect(readonly=True) as con:
    m = read(con, "SELECT CAST(strftime('%m', date) AS INTEGER) month, "
                  "AVG(precip_mm) mm FROM weather_daily GROUP BY month")
    yr = read(con, "SELECT CAST(strftime('%Y', date) AS INTEGER) y, "
                   "AVG(precip_mm)*365 mm FROM weather_daily GROUP BY y")

fig, ax = plt.subplots(1, 2, figsize=(13, 3.4))
ax[0].bar(m.month, m.mm, color=['#9aa7b1']*5 + ['#2b7fd4']*4 + ['#9aa7b1']*3)
ax[0].set_title('mean daily rainfall by month'); ax[0].set_xticks(range(1, 13))
ax[1].bar(yr.y, yr.mm, color='#2b7fd4')
ax[1].set_title('approx annual rainfall (mm) — Maharashtra ~600–2500')
plt.tight_layout(); plt.show()
print('peak month:', int(m.loc[m.mm.idxmax(), 'month']), '(must be 6–9)')

## 3. Does the interpolated curve look real?

This is the Sat 12:00 sync-point decision. You are looking for:

* sharp **recharge** during the monsoon (level rises — the line moves *up* on
  the inverted axis),
* smooth **recession** through the dry season,
* the curve landing **exactly** on every red dot.

If it looks like straight lines between dots with no rainfall response,
tell Dev A. If it cannot be fixed in an hour, the honest fallback is linear +
rainfall bump — ugly, defensible, ships.

In [ ]:
WELL = None   # set a well_id, or leave None for the one with most data

with connect(readonly=True) as con:
    if WELL is None:
        WELL = con.execute('SELECT well_id FROM gw_daily GROUP BY well_id '
                           'ORDER BY COUNT(*) DESC LIMIT 1').fetchone()[0]
    g = read(con, 'SELECT date, level_mbgl, is_observed, confidence FROM gw_daily '
                  'WHERE well_id=? ORDER BY date', (WELL,))
    w = read(con, 'SELECT date, precip_mm FROM weather_daily WHERE well_id=? '
                  'ORDER BY date', (WELL,))
g['date'] = pd.to_datetime(g.date); w['date'] = pd.to_datetime(w.date)

fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True,
                       gridspec_kw={'height_ratios': [3, 1]})
ax[0].plot(g.date, g.level_mbgl, lw=.9, color='#2b7fd4', label='interpolated daily')
o = g[g.is_observed == 1]
ax[0].scatter(o.date, o.level_mbgl, s=28, color='#d94b2b', zorder=5,
              label=f'real readings (n={len(o)})')
ax[0].invert_yaxis(); ax[0].set_ylabel('m below ground'); ax[0].legend(fontsize=8)
ax[0].set_title(f'{WELL} — does this look like groundwater?')
ax[1].bar(w.date, w.precip_mm, width=1.0, color='#6aa9e0')
ax[1].set_ylabel('rain mm')
plt.tight_layout(); plt.show()

print(f'{len(o)} real of {len(g)} daily rows = {100*len(o)/len(g):.2f}% observed')

## 4. Leakage audit

The single most likely reason a judge stops believing your accuracy number.

In [ ]:
with connect(readonly=True) as con:
    sp = read(con, 'SELECT split, COUNT(*) n, MIN(date) first, MAX(date) last, '
                   'COUNT(DISTINCT entity_id) entities FROM features GROUP BY split')
display(sp.sort_values('first'))

s = sp.set_index('split')
order = [x for x in ['train', 'val', 'test'] if x in s.index]
leak = any(s.loc[order[i], 'last'] >= s.loc[order[i+1], 'first']
           for i in range(len(order)-1))
print('OVERLAP DETECTED — this is leakage' if leak
      else 'No overlap. Splits are chronological.')
print('scenario date', cfg['scenario_date'], 'is in the',
      s.index[(s['first'] <= str(cfg['scenario_date'])) &
              (s['last'] >= str(cfg['scenario_date']))].tolist(), 'split')

## 5. Urban track — the demo contrast

In [ ]:
with connect(readonly=True) as con:
    r = read(con, "SELECT date, live_storage_pct, source FROM reservoir_daily "
                  "WHERE reservoir_id='MUM_ALL' ORDER BY date")
r['date'] = pd.to_datetime(r.date)
real = r[r.source.isin(['manual', 'wrd_pravah'])]

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(r.date, r.live_storage_pct, lw=1.2, color='#9aa7b1', label='interpolated')
ax.scatter(real.date, real.live_storage_pct, s=34, color='#d94b2b', zorder=5,
           label=f'published readings (n={len(real)})')
ax.axvline(pd.Timestamp(cfg['scenario_date']), ls='--', lw=.9, color='#444')
ax.set_ylabel('% live storage'); ax.legend(fontsize=8)
ax.set_title('Mumbai lakes 2026 — scenario date marked')
plt.tight_layout(); plt.show()
display(real)

## 6. Generate the data card

Pass Dev A's held-out interpolation MAE (task A5) so it lands in the card.

In [ ]:
!python tools/data_card.py --mae 0.00   # <-- replace with A's real MAE